# Level 3 – Task 2: NLP – Text Classification (Sentiment Analysis)
**Dataset:** Social Media Sentiment Dataset  
**Goal:** Classify posts as Positive, Negative, or Neutral using NLP techniques.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

sns.set_style("whitegrid")

df = pd.read_csv('../3) Sentiment dataset.csv')
print("Raw dataset shape:", df.shape)
print("Unique sentiment labels:", df['Sentiment'].nunique())
df[['Text', 'Sentiment']].head(3)

Raw dataset shape: (732, 15)
Unique sentiment labels: 279


,Text,Sentiment
0,Enjoying a beautiful day at the park! ...,Positive
1,Traffic was terrible this morning. ...,Negative
2,Just finished an amazing workout! 💪 ...,Positive


## 1. Group 279 Fine-Grained Labels → Positive / Negative / Neutral

In [2]:
# The dataset has 279 micro-labels (e.g. "Elation", "Melancholy", "Nostalgia")
# Standard NLP practice: map them to 3 broad sentiment classes

POSITIVE = {
    'positive','joy','excitement','happy','happiness','elation','playful',
    'serenity','contentment','gratitude','hopeful','empowerment','inspired',
    'determination','love','amazing','wonderful','cheerful','delight',
    'enthusiasm','pride','relief','bliss','euphoria','optimism','motivated',
    'affection','passion','admiration','grateful','joyful','energized',
    'hope','comfort','pleased','satisfied','ecstatic','confident',
    'appreciation','laughter','amusement','curiosity','creative'
}
NEGATIVE = {
    'negative','sad','sadness','despair','bad','hate','anger','angry',
    'fear','disgust','stressed','stress','anxiety','anxious','depressed',
    'depression','loneliness','lonely','melancholy','numbness','grief',
    'frustration','frustrated','disappointment','disappointed','upset',
    'worry','worried','hopeless','regret','bitterness','resentment',
    'shame','guilt','hurt','pain','sorrow','misery','heartbreak',
    'jealousy','envy','hatred','boredom','embarrassed'
}

def map_sentiment(label):
    l = str(label).lower().strip()
    if l in POSITIVE:
        return 'Positive'
    if l in NEGATIVE:
        return 'Negative'
    return 'Neutral'

df['Sentiment'] = df['Sentiment'].apply(map_sentiment)

print("Sentiment distribution after grouping:")
print(df['Sentiment'].value_counts())


Sentiment distribution after grouping:
Sentiment
Positive    300
Neutral     299
Negative    133
Name: count, dtype: int64


## 2. Visualize Class Distribution

In [3]:
plt.figure(figsize=(6, 4))
df['Sentiment'].value_counts().plot(
    kind='bar', color=['green', 'red', 'grey'], edgecolor='black')
plt.title("Sentiment Class Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('dist.png', dpi=80)
plt.show()

for s in df['Sentiment'].unique():
    print(f"[{s}]: {df[df['Sentiment']==s]['Text'].iloc[0][:100]}")


[Positive]:  Enjoying a beautiful day at the park!              
[Negative]:  Traffic was terrible this morning.                 
[Neutral]:  Trying out a new recipe for dinner tonight.        


## 3. Text Preprocessing

In [4]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)   # remove URLs
    text = re.sub(r'@\w+|#\w+', '', text)         # remove mentions/hashtags
    text = re.sub(r'[^a-z\s]', '', text)            # keep only letters
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

df['clean_text'] = df['Text'].apply(clean_text)

print("Original:", df['Text'].iloc[0])
print("Cleaned: ", df['clean_text'].iloc[0])


Original:  Enjoying a beautiful day at the park!              
Cleaned:  enjoying beautiful day park


## 4. TF-IDF Vectorization

In [5]:
# TF-IDF: score each word by how unique it is to a document vs all documents
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))

X = tfidf.fit_transform(df['clean_text'])
y = df['Sentiment']

print("Feature matrix shape:", X.shape)
print("Classes:", y.unique().tolist())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")


Feature matrix shape: (732, 3000)
Classes: ['Positive', 'Negative', 'Neutral']
Train: 585  |  Test: 147


## 5. Train Naive Bayes

In [6]:
nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)

print("Naive Bayes Accuracy:", round(accuracy_score(y_test, nb_pred), 4))
print()
print(classification_report(y_test, nb_pred))


Naive Bayes Accuracy: 0.619

              precision    recall  f1-score   support

    Negative       1.00      0.14      0.25        28
     Neutral       0.61      0.72      0.66        61
    Positive       0.61      0.74      0.67        58

    accuracy                           0.62       147
   macro avg       0.74      0.54      0.53       147
weighted avg       0.68      0.62      0.59       147



## 6. Train Logistic Regression

In [7]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

print("Logistic Regression Accuracy:", round(accuracy_score(y_test, lr_pred), 4))
print()
print(classification_report(y_test, lr_pred))


Logistic Regression Accuracy: 0.6327

              precision    recall  f1-score   support

    Negative       1.00      0.14      0.25        28
     Neutral       0.60      0.75      0.67        61
    Positive       0.65      0.74      0.69        58

    accuracy                           0.63       147
   macro avg       0.75      0.55      0.54       147
weighted avg       0.70      0.63      0.60       147



## 7. Model Comparison

In [8]:
results = pd.DataFrame({
    'Model':          ['Naive Bayes', 'Logistic Regression'],
    'Accuracy':       [accuracy_score(y_test, nb_pred),
                       accuracy_score(y_test, lr_pred)],
    'F1 (weighted)':  [f1_score(y_test, nb_pred, average='weighted'),
                       f1_score(y_test, lr_pred, average='weighted')]
}).round(4)

print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 4))
results.set_index('Model')['Accuracy'].plot(
    kind='bar', color=['steelblue','salmon'], edgecolor='black', ax=ax)
ax.set_title("Model Accuracy Comparison")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
fig


              Model  Accuracy  F1 (weighted)
        Naive Bayes    0.6190         0.5852
Logistic Regression    0.6327         0.5979


<Figure size 600x400 with 1 Axes>

## 8. Confusion Matrix

In [9]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
classes = sorted(y.unique())

for ax, (name, pred) in zip([ax1, ax2],
                             [('Naive Bayes', nb_pred),
                              ('Logistic Regression', lr_pred)]):
    cm = confusion_matrix(y_test, pred, labels=classes)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=classes, yticklabels=classes)
    ax.set_title(name)
    ax.set_ylabel("Actual")
    ax.set_xlabel("Predicted")

plt.suptitle("Confusion Matrices - Sentiment Classification", fontsize=13)
plt.tight_layout()
fig


<Figure size 1200x400 with 4 Axes>

## 9. Top Words Per Sentiment Class

In [10]:
feature_names = tfidf.get_feature_names_out()

for i, cls in enumerate(lr.classes_):
    top_idx = np.argsort(lr.coef_[i])[-10:][::-1]
    top_words = [feature_names[j] for j in top_idx]
    print(f"[{cls}] top words: {', '.join(top_words)}")


[Negative] top words: despair, grief, loneliness, lingers, away, injustice, accidentally, regret, feeling, resentment
[Neutral] top words: compassion, sky, warmth, tenderness, confusion, resonates, calmness, stone, accomplishment, embracing
[Positive] top words: new, gratitude, friend, serenity, contentment, hopeful, pride, determination, joined, curiosity


## Summary
- Grouped 279 raw labels → 3 standard sentiment classes (Positive / Negative / Neutral)
- Cleaned text: lowercased, removed noise, lemmatized, removed stopwords
- TF-IDF vectorization with bigrams
- **Logistic Regression outperforms Naive Bayes** on this dataset
- Top words per class align well with human intuition